# Deep Learning Baseline: RoBERTa

This notebook explores a deep learning baseline for the ICD-10 codification task. 
Instead of relying on TF-IDF and traditional machine learning, we fine-tune a pre-trained biomedical language model.

Specifically, we use `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es`, which is well-suited for Spanish clinical text.
The goal is to determine whether dense contextual embeddings and a neural classification head can outperform our lexical TF-IDF baselines on short clinical literals.

> **Archive note.** This notebook is kept as part of the project history. It shows an earlier stage of our thinking before the final repository structure was cleaned up. The final reproducible story and current decisions are in notebooks `00` to `08`, the scripts under `src/` and `models/`, and the final report.


In [1]:
import os
import sys
import time
import warnings
from pathlib import Path
import re

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

sys.path.insert(0, os.path.abspath('../src'))

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATA_DIR = Path('../data')
SUBMISSION_DIR = Path('../submissions')
MODELS_DIR = Path('../models')
SUBMISSION_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print('Setup complete.')

Using device: cpu
Setup complete.


## 1. Load and preprocess the data

We load the supervised training pairs and the leaderboard literals. 
For the deep learning model, we apply a very light preprocessing step to preserve casing, accents, and punctuation, which might carry semantic value for a pre-trained language model.

In [2]:
codif_df = pd.read_csv(DATA_DIR / 'codification_data.csv')
lead_df = pd.read_csv(DATA_DIR / 'leaderboard_data.csv')

def preprocess_dl(text):
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

codif_df['Literal'] = codif_df['Literal'].apply(preprocess_dl)
lead_df['Literal'] = lead_df['Literal'].apply(preprocess_dl)

print(f'Training rows:   {len(codif_df):,}')
print(f'Leaderboard:     {len(lead_df):,}')

display(codif_df.head())

Training rows:   13,700
Leaderboard:     6,667


,Code,Literal
0,J9809,Hiperreactividad bronquial
1,J9801,broncoespástica
2,I420,miocardiopatía dilatada
3,Y831,HTA irc 6
4,R5600,Crisis febriles atípicas


## 2. Prepare the target variable

The target is the first character of the ICD code. We extract this category and map it to an integer index for the PyTorch cross-entropy loss. We then create an 80/20 stratified split for training and validation.

In [3]:
codif_df['y_category'] = codif_df['Code'].astype(str).str[0]
categories = sorted(codif_df['y_category'].unique())
cat2idx = {cat: i for i, cat in enumerate(categories)}
idx2cat = {i: cat for i, cat in enumerate(categories)}

codif_df['label'] = codif_df['y_category'].map(cat2idx)

X_train, X_val, y_train, y_val = train_test_split(
    codif_df['Literal'].values, 
    codif_df['label'].values, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=codif_df['label'].values
)

print(f"Categories: {len(categories)}")
print(f"Train size: {len(X_train):,}")
print(f"Val size:   {len(X_val):,}")

Categories: 36
Train size: 10,960
Val size:   2,740


## 3. Dataset and DataLoader

We define a custom PyTorch `Dataset` that tokenizes the literals using the RoBERTa tokenizer.

In [4]:
class ICD10Dataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        sample = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        if self.labels is not None:
            sample['labels'] = torch.tensor(self.labels[item], dtype=torch.long)
        return sample

model_name = 'PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'
tokenizer = AutoTokenizer.from_pretrained(model_name)

batch_size = 128
train_dataset = ICD10Dataset(X_train, y_train, tokenizer)
val_dataset = ICD10Dataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

## 4. Model Architecture

We construct a classification head on top of the RoBERTa base model. 
To extract features from the token embeddings, we apply **Mean Pooling** (ignoring padding tokens), followed by a dropout layer and a linear classifier mapping to our 36 categories.

In [5]:
class RobertaClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super(RobertaClassifier, self).__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.drop = nn.Dropout(0.1)
        self.out = nn.Linear(self.roberta.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        
        # Mean pooling ignoring padding tokens
        last_hidden_state = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        
        dropped = self.drop(mean_pooled)
        return self.out(dropped)

model = RobertaClassifier(model_name, num_classes=len(categories))
model = model.to(device)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 49647.17it/s]
[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. Training Loop

We train the model using `AdamW` and `CrossEntropyLoss`. The training process incorporates early stopping based on validation accuracy to prevent overfitting. We aim to reach or exceed a validation accuracy of 0.570.

In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()

epochs = 50
patience = 10
best_acc = 0.0
epochs_no_improve = 0
best_model_path = MODELS_DIR / 'best_roberta_baseline.pt'

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    # Validation
    model.eval()
    val_loss = 0
    preds = []
    true_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            preds.extend(predicted.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
            
    val_acc = accuracy_score(true_labels, preds)
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        epochs_no_improve += 1
        
    if epochs_no_improve >= patience:
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

print(f"Best Validation Accuracy: {best_acc:.4f}")

AttributeError: RobertaTokenizer has no attribute encode_plus

## 6. Inference and Submission

Finally, we load the best weights, generate predictions for the leaderboard dataset, and save the results for submission.

In [ ]:
# Load the best model
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_dataset = ICD10Dataset(lead_df['Literal'].values, labels=None, tokenizer=tokenizer)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

test_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, 1)
        test_preds.extend(predicted.cpu().numpy())

lead_df['y_category'] = [idx2cat[p] for p in test_preds]
output_path = SUBMISSION_DIR / 'roberta_baseline_predictions.csv'

lead_df[['id', 'y_category']].to_csv(output_path, index=False)
print(f"Predictions successfully saved to {output_path}")